<a href="https://colab.research.google.com/github/LSHummel/1CCPG-IA-FIAP-2026/blob/main/2Sem_Aula_01_Revisao_LangChain_LCEL_ChatOllama_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Notebook do Professor (Demo) — Aula 01: Revisão expressa + LangChain LCEL e ChatOllama

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 01/14 — Módulo 1: LangChain Foundations**  
**⏱️ 1h40min**  
**🐍 LCEL · ChatOllama · OutputParser**  
**🔁 Andaime 40%**  

---

## 🎯 Objetivo da aula

Construir uma chain LangChain com LCEL que substitui o chamar_llm() manual do 1º semestre — com menos código, mais composição e pronto para escalar com memória e RAG nas próximas aulas.

---

## Como usar este notebook

- Cada célula corresponde a um slide de código da aula (a ordem é a da apresentação).
- Rode ao vivo enquanto explica o slide correspondente.
- A última seção traz as soluções dos exercícios para executar em sala.

---

# 🔬 Código da aula — slide a slide

### Slide 07 — O ponto de partida: o chatbot manual do 1º semestre

In [ ]:
!pip install langchain-ollama langchain-core langchain-classic -q

from langchain_ollama import ChatOllama
from google.colab import userdata
import os

# Definir a API key via variável de ambiente (Colab Secrets)
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
def montar_prompt(historico, pergunta):
    return [{"role": "system", "content": SYSTEM_PROMPT}] \
        + historico + [{"role": "user", "content": pergunta}]

def chamar_llm(messages):
    resp = ollama.chat(model="gpt-oss:120b", messages=messages)
    return resp["message"]["content"]

def iniciar_chat():
    historico = []
    while True:
        pergunta = input("Você: ")
        if pergunta == "sair": break
        msgs = montar_prompt(historico, pergunta)
        resposta = chamar_llm(msgs)
        historico += [{"role": "user", "content": pergunta},
                     {"role": "assistant", "content": resposta}]
        print("Bot:", resposta)

### Slide 11 — LCEL — o operador | como pipe

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Cada componente é independente — compostos pelo operador |

# 1. Template: define estrutura e variáveis do prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "Você é {persona}. Responda sobre {especialidade}."),
    ("human",  "{pergunta}"),
])

# 2. Modelo: conecta ao Ollama Cloud
llm = ChatOllama(
    model="gpt-oss:120b",
    base_url="https://ollama.com",
)

# 3. Parser: extrai só o texto da resposta
parser = StrOutputParser()

# Composição com |  — isso é uma Runnable, não uma chamada
chain = prompt | llm | parser

# Invocar a chain com as variáveis do template
resposta = chain.invoke({
    "persona":       "um chef de culinária brasileira",
    "especialidade": "culinária brasileira",
    "pergunta":      "Como faço um bolo de cenoura?",
})
print(resposta)  # → string direta, sem resp["message"]["content"]

### Slide 12 — ChatPromptTemplate — evolução do montar_prompt()

In [ ]:
def montar_prompt(pergunta,
                    especialidade,
                    tom="direto"):
    return f"""
Você é especialista em {especialidade}.
Pergunta: {pergunta}
Tom: {tom}.
"""

# Problema: f-string não valida variáveis
# se esquecer uma, silencia sem erro

### Slide 12 — ChatPromptTemplate — evolução do montar_prompt()

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Você é especialista em {especialidade}."
     " Tom: {tom}."),
    ("human", "Pergunta: {pergunta}"),
])

# Vantagem: valida variáveis automaticamente
# KeyError se esquecer alguma ao invocar
# Separação de roles limpa (system/human/ai)

### Slide 13 — Output Parsers — StrOutputParser e JsonOutputParser

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

# Retorna string direta
resposta = chain.invoke({"pergunta":"Oi"})
print(type(resposta))  # → <class 'str'>

# Sem mais resp["message"]["content"] !

### Slide 13 — Output Parsers — StrOutputParser e JsonOutputParser

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

chain = prompt | llm | JsonOutputParser()

# O prompt DEVE pedir JSON explicitamente
resultado = chain.invoke({
    "pergunta": "Liste 3 ingredientes"
})
print(type(resultado))   # → <class 'dict'>
print(resultado["ingredientes"])

### Slide 14 — ChatOllama — conectando ao Ollama Cloud

In [ ]:
!pip install langchain-ollama langchain-core langchain-classic -q

from langchain_ollama import ChatOllama
from google.colab import userdata
import os

# Definir a API key via variável de ambiente (Colab Secrets)
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

# Instanciar o modelo — parâmetros declarativos
llm = ChatOllama(
    model="gpt-oss:120b",          # modelo padrão do semestre
    temperature=0.7,               # mesmo parâmetro do 1º sem
    num_predict=512,               # equivalente a max_tokens
)

# Teste rápido direto (sem chain)
resp = llm.invoke("Olá! Responda em 1 linha.")
print(resp.content)   # .content — não ["message"]["content"]
print(type(resp))      # → AIMessage (objeto LangChain)

### Slide 24 — Resumo — o que é novo no Python desta aula

In [ ]:
# 1. Instância declarativa com keyword arguments
llm = ChatOllama(model="gpt-oss:120b", temperature=0.7)

# 2. Método de classe com lista de tuplas
prompt = ChatPromptTemplate.from_messages([
    ("system", "Você é {persona}."),
    ("human",  "{pergunta}"),
])

# 3. Operator overloading — | cria chain composta
chain = prompt | llm | StrOutputParser()

# 4. .invoke() — método com dict de variáveis
resp = chain.invoke({"persona": "chef", "pergunta": "Olá"})

# 5. Generator com for + yield (em .stream())
for chunk in chain.stream({"pergunta": "Oi"}):
    print(chunk, end="", flush=True)
    # flush=True força exibição imediata de cada token

# 6. os.environ — variáveis de ambiente em Python
import os
os.environ["OLLAMA_API_KEY"] = "valor"  # dict especial do SO

---

## 🏋️ Exercícios Resolvidos — versão professor (executar no Colab)

As quatro soluções prontas dos exercícios de fixação do notebook do aluno — rode em sala, uma a uma.


### Exercício 1 — Monte a chain e inspecione os tipos

**Para o professor:** a solução completa o andaime — template do domínio, chain na ordem certa — e imprime o tipo de saída de cada estágio: mensagens no prompt, `AIMessage` no modelo e `str` na chain. Feche com o `KeyError` provocado no `.invoke()` incompleto: o template valida as variáveis, a f-string antiga falhava em silêncio.


In [ ]:
# Setup da aula — imports e conexão com o Ollama Cloud (Colab Secrets)
!pip install langchain-ollama langchain-core langchain-classic -q

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from google.colab import userdata
import os

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

# 1) Role e persona do domínio no system; {pergunta} no human
prompt = ChatPromptTemplate.from_messages([
    ("system", "Você é um guia de {dominio}. Responda em 1 frase."),
    ("human",  "{pergunta}"),
])

# 2) Chain na ordem certa: prompt → modelo → parser
llm = ChatOllama(model="gpt-oss:120b", temperature=0.7)
chain = prompt | llm | StrOutputParser()

variaveis = {"dominio": "culinária", "pergunta": "Como conservar manjericão?"}

# 3) Tipos de cada estágio
mensagens = prompt.invoke(variaveis)
print("prompt →", type(mensagens).__name__)
print("modelo →", type(llm.invoke(mensagens.to_messages())).__name__)   # AIMessage
resposta = chain.invoke(variaveis)
print("chain  →", type(resposta))          # <class 'str'>
print(resposta)

# 4) Invoke incompleto → KeyError imediato (o template valida as variáveis)
try:
    chain.invoke({"dominio": "culinária"})
except KeyError as e:
    print("KeyError:", e)


### Exercício 2 — Chain quebrada: análise e correção no código

**Para o professor:** a célula roda primeiro a versão com defeitos (composição invertida, parser ausente e chave `pergunta` ausente) para exibir o sintoma, e depois a corrigida — `prompt | llm | StrOutputParser()` — com `<class 'str'>` confirmado no `print(type(resposta))`. Destaque: o prompt vem antes do modelo porque é ele quem transforma o dict em mensagens.


In [ ]:
# Setup da aula — imports e conexão com o Ollama Cloud (Colab Secrets)
!pip install langchain-ollama langchain-core langchain-classic -q

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from google.colab import userdata
import os

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

# Versão com defeitos: 3 problemas para analisar antes de corrigir
llm = ChatOllama(model="gpt-oss:120b", temperature=0.7)

prompt_errado = ChatPromptTemplate.from_messages([
    ("system", "Você é especialista em {tema}. Responda em 2 frases."),
    ("human",  "{pergunta}"),
])
try:
    chain_errada = llm | prompt_errado          # (a) ordem invertida
    chain_errada.invoke({"tema": "culinária"})  # (c) falta "pergunta" — (b) parser ausente
except Exception as e:
    print(type(e).__name__, "→", str(e)[:140])

# Versão corrigida: prompt → modelo → parser, todas as variáveis no invoke
prompt = ChatPromptTemplate.from_messages([
    ("system", "Você é especialista em {tema}. Responda em 2 frases."),
    ("human",  "{pergunta}"),
])
chain = prompt | llm | StrOutputParser()

resposta = chain.invoke({"tema": "culinária", "pergunta": "Qual o segredo de um bom caldo?"})

print(type(resposta))   # → <class 'str'>
print(resposta)


### Exercício 3 — Temperature e parsers: compare no código

**Para o professor:** a solução roda a mesma pergunta com `temperature=0` e `0.9` duas vezes cada — o contraste entre saída quase determinística e variações criativas aparece na tela. Em seguida completa a instrução de formato no system e o `JsonOutputParser()`, confirmando `dict` com as chaves `resposta` e `nivel`. Vale pausar entre as execuções para discutir reprodutibilidade.


In [ ]:
# Setup da aula — imports e conexão com o Ollama Cloud (Colab Secrets)
!pip install langchain-ollama langchain-core langchain-classic -q

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from google.colab import userdata
import os

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

prompt_base = ChatPromptTemplate.from_messages([
    ("system", "Você é um guia de {dominio}. Responda em 1 frase."),
    ("human",  "{pergunta}"),
])

# temperature=0 → saída quase determinística; 0.9 → varia entre execuções
chain_fria     = prompt_base | ChatOllama(model="gpt-oss:120b", temperature=0.0) | StrOutputParser()
chain_criativa = prompt_base | ChatOllama(model="gpt-oss:120b", temperature=0.9) | StrOutputParser()

for i in range(2):
    print("fria:    ", chain_fria.invoke({"dominio": "culinária", "pergunta": "Como conservar manjericão?"}))
    print("criativa:", chain_criativa.invoke({"dominio": "culinária", "pergunta": "Como conservar manjericão?"}))
    print()

# O JsonOutputParser é sintático: a instrução no prompt garante as chaves,
# o format="json" garante JSON válido — o resultado chega como dict
prompt_json = ChatPromptTemplate.from_messages([
    ("system", "Você é um guia de {dominio}. Responda SOMENTE em JSON com as chaves 'resposta' e 'nivel'."),
    ("human",  "{pergunta}"),
])
chain_json = prompt_json | ChatOllama(model="gpt-oss:120b", format="json") | JsonOutputParser()

r = chain_json.invoke({"dominio": "culinária", "pergunta": "Um lanche rápido"})
print(type(r))          # → <class 'dict'>
print(list(r.keys()))   # → ['resposta', 'nivel'] (ordem pode variar)
print(r)


### Exercício 4 — Chain JSON do domínio do grupo

**Para o professor:** a solução fecha o ciclo com o pipeline do domínio — persona no system, instrução de formato com as chaves `resumo`/`topicos`/`nivel`, `format="json"` e `JsonOutputParser()` na ponta. O `list(r.keys())` na saída confirma o dict com as chaves pedidas — ponte direta para o Pydantic da Aula 03, que valida o schema por inteiro.


In [ ]:
# Setup da aula — imports e conexão com o Ollama Cloud (Colab Secrets)
!pip install langchain-ollama langchain-core langchain-classic -q

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from google.colab import userdata
import os

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

prompt_json = ChatPromptTemplate.from_messages([
    ("system",
     "Você é um chef consultor de culinária brasileira. "
     "Responda SOMENTE em JSON com as chaves 'resumo' (texto curto), "
     "'topicos' (lista com exatamente 3 itens) e 'nivel' "
     "(um de: iniciante, intermediario, avancado)."),
    ("human", "{pergunta}"),
])

chain_json = prompt_json | ChatOllama(model="gpt-oss:120b", format="json") | JsonOutputParser()

perguntas = [
    "Como fazer um bom feijão tropeiro?",
    "Qual vinho harmonizo com uma moqueca capixaba?",
]
for pergunta in perguntas:
    r = chain_json.invoke({"pergunta": pergunta})
    print(type(r))           # → <class 'dict'>
    print(list(r.keys()))    # → ['resumo', 'topicos', 'nivel'] (ordem pode variar)
    print(r)

# Bônus — .stream() entrega pedaços conforme gerados; em JSON, os pedaços
# só formam um dict válido no final — acumule e parseie depois.
acumulado = ""
for chunk in (prompt_json | ChatOllama(model="gpt-oss:120b", format="json")).stream({"pergunta": perguntas[0]}):
    acumulado += chunk.content
print("Stream acumulado:", acumulado[:80], "...")


## 📚 Referências da aula

- Docs LangChain — LCEL (LangChain Expression Language): composição declarativa de chains com o operador |. python.langchain.com/docs/concepts/lcel
- Docs LangChain — ChatPromptTemplate: estruturar prompts com roles e variáveis. python.langchain.com/docs/concepts/prompt_templates
- Docs LangChain — Output parsers: StrOutputParser, JsonOutputParser, PydanticOutputParser. python.langchain.com/docs/concepts/output_parsers
- Docs langchain-ollama — ChatOllama: integração LangChain com modelos Ollama. python.langchain.com/docs/integrations/chat/ollama
- Livro Russell, S.; Norvig, P. — Inteligência Artificial. 3ª ed. Pearson, 2016. Cap. 2: Agentes racionais — o modelo de percepção → ação que fundamenta o conceito de pipeline de LLM.
- Ebook Polzer, D. — RAG with Python Cookbook. O'Reilly, 2025. Cap. 1: escolha de frameworks para aplicações RAG e por que LangChain (LCEL) é o padrão de mercado para orquestração.
- Ebook Gullí, A. — Agentic Design Patterns. O'Reilly, 2025. Cap. 1: Prompt Chaining — a base dos padrões agênticos; o LCEL que você aprendeu hoje é a fundação dos agentes da Aula 10.
- Ebook Lanham, M. — AI Agents in Action. Manning, 2025. Cap. 2: prompting LLMs com personas e delimitadores — reforça o que você construiu no 1º semestre.

---

**→ Próxima Aula — Aula 02 · 10/08** — Memória conversacional — Buffer, Summary e TokenBuffer
  
A chain de hoje é stateless — cada .invoke() começa do zero, igual à lista historico manual do 1º semestre, que crescia sem limite de tokens. A Aula 02 resolve isso com 3 tipos de memória — Buffer, Summary e TokenBuffer — e o CKP01 fica mais próximo.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*